# Phase 4: Report Assembly
Merges Phase 1+2+3 outputs into validated CertificationReport.

In [5]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
from scripts.report_assembler import ReportAssembler

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'
phase2_path = NB_DATA / 'computed_content.json'
phase3_path = NB_DATA / 'narratives.json'

assembler = ReportAssembler(phase1_path, phase2_path, phase3_path)
report = assembler.assemble()

# Save to notebook-local data
(NB_DATA / 'certification_report.json').write_text(
    json.dumps(report, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print(f'Agent: {report["meta"]["agent_name"]}')
print(f'Date:  {report["meta"]["certification_date"]}')
print(f'Runs:  {report["meta"]["total_runs"]}')

print(f'\n=== Sections ({len(report["sections"])}) ===')
for s in report['sections']:
    blocks = len(s.get('content', []))
    part = f' [{s["part"]}]' if s.get('part') else ''
    print(f'  {s["number"]:2d}. {s["title"]}{part} ({blocks} blocks)')

print(f'\n=== Header ===')
print(f'Scorecard: {len(report["header"]["scorecard"])} dimensions')
print(f'Findings:  {len(report["header"]["findings"])} items')

print(f'\nFooter: {report["footer"]}')

---
## schema additions for statistical findings

Confirm three new content blocks (`NoticeBlock`, `HypothesisStripBlock`, `CIBarChartBlock`) validate, and that all existing certification reports still validate against the extended schema.

In [6]:
# Construct one of each new block from a literal dict and validate.
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from schema.certification_schema import (
    NoticeBlock,
    HypothesisStripBlock,
    CIBarChartBlock,
    NoticeSeverity,
)

notice = NoticeBlock.model_validate({
    "type": "notice",
    "severity": "warning",
    "title": "Statistical hypothesis testing skipped",
    "body": "Ground-truth directory not supplied; hypothesis framework was skipped.",
})
print("NoticeBlock           :", notice.model_dump())

hyp_strip = HypothesisStripBlock.model_validate({
    "type": "hypothesis_strip",
    "verdict": "flag",
    "hypothesis_id": "H-01",
    "summary": "TTD wide CI on Resource (\u00b155s, 50% of IQM); driven by pod-network-corruption (n=24).",
    "detail": "Recommend +30 runs in Resource category to tighten interval.",
})
print("HypothesisStripBlock  :", hyp_strip.verdict, hyp_strip.hypothesis_id)

# CI bar chart with `group` field: paired Detection / Mitigation whiskers per category.
ci_chart = CIBarChartBlock.model_validate({
    "type": "chart",
    "chart_type": "ci_bar",
    "title": "Detection & Mitigation rate by fault category (Wilson 95% CI)",
    "y_label": "rate",
    "points": [
        {"label": "application_fault", "group": "Detection",  "value": 0.94, "ci_low": 0.85, "ci_high": 0.98},
        {"label": "application_fault", "group": "Mitigation", "value": 0.88, "ci_low": 0.78, "ci_high": 0.94},
        {"label": "network_fault",     "group": "Detection",  "value": 0.91, "ci_low": 0.81, "ci_high": 0.96},
        {"label": "network_fault",     "group": "Mitigation", "value": 0.83, "ci_low": 0.72, "ci_high": 0.91},
        {"label": "resource_fault",    "group": "Detection",  "value": 0.79, "ci_low": 0.66, "ci_high": 0.88},
        {"label": "resource_fault",    "group": "Mitigation", "value": 0.71, "ci_low": 0.58, "ci_high": 0.82},
    ],
})
print("CIBarChartBlock       :", ci_chart.chart_type, "n=", len(ci_chart.points), "groups=", sorted({p.group for p in ci_chart.points}))

NoticeBlock           : {'type': 'notice', 'severity': <NoticeSeverity.warning: 'warning'>, 'title': 'Statistical hypothesis testing skipped', 'body': 'Ground-truth directory not supplied; hypothesis framework was skipped.'}
HypothesisStripBlock  : flag H-01
CIBarChartBlock       : ci_bar n= 6 groups= ['Detection', 'Mitigation']


In [7]:
# JSON round-trip: dump -> validate -> equal.
for blk in (notice, hyp_strip, ci_chart):
    s = blk.model_dump_json()
    rt = type(blk).model_validate_json(s)
    assert rt == blk, f"round-trip mismatch for {type(blk).__name__}"
    print(f"  OK round-trip {type(blk).__name__:<22} ({len(s)} chars)")

  OK round-trip NoticeBlock            (167 chars)
  OK round-trip HypothesisStripBlock   (236 chars)
  OK round-trip CIBarChartBlock        (715 chars)


In [8]:
# Backwards-compat: existing certification reports must still validate
# against the extended schema (no new required fields anywhere).
import json as _json
from schema.certification_schema import CertificationReport

REPO = ROOT
candidates = [
    REPO / 'data_certifier' / 'scenario_1_out' / 'certification_report_sre-agent-v2.1.json',
    REPO / 'data_certifier' / 'scenario_2_out' / 'certification_report_sre-agent-v2.1.json',
]

for p in candidates:
    if not p.exists():
        print(f"  SKIP (missing) {p}")
        continue
    rep = CertificationReport.model_validate(_json.loads(p.read_text(encoding='utf-8')))
    print(f"  PASS {p.name}: {len(rep.sections)} sections, agent={rep.meta.agent_name}")

  PASS certification_report_sre-agent-v2.1.json: 12 sections, agent=SRE-Agent
  PASS certification_report_sre-agent-v2.1.json: 12 sections, agent=SRE-Agent


## Sample-size notice (skip-path)

When `runs_per_fault < 30`, §1 of the assembled report appends a hard-coded
`NoticeBlock(severity="warning")` informing the reader that the
9-Hypothesis Statistical Framework was not evaluated. The notice text is a
single string constant — no per-category breakdown, no number interpolation.
The wiring lives in `cert_builder/scripts/report_assembler.py::_section_executive_summary`.

In [16]:
import importlib
import sys
from pathlib import Path

# The narratives package __init__ uses `cert_builder.*` absolute imports,
# so the parent of cert_builder must be on sys.path.
_REPO_ROOT = ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import cert_builder.scripts.narratives.sample_size_notice_builder as _ssnb
importlib.reload(_ssnb)
build_sample_size_notice = _ssnb.build_sample_size_notice
MIN_RUNS_PER_FAULT = _ssnb.MIN_RUNS_PER_FAULT

from cert_builder.schema.certification_schema import NoticeBlock as _NoticeBlock

# Phase D — sample-size notice builder
# The notice now gates on the upstream statistical_hypothesis.status:
#   "skipped"       -> emit notice (framework requested but blocked)
#   "ok"            -> no notice (framework ran successfully)
#   "not_requested" -> no notice (e.g., run without --advanced-analysis)
#   missing / None  -> no notice
assert build_sample_size_notice(None) is None, "missing block must NOT emit"
assert build_sample_size_notice({"status": "not_requested"}) is None, \
    "framework not requested must NOT emit (no --advanced-analysis case)"
assert build_sample_size_notice({"status": "ok", "results": {}}) is None, \
    "framework ran successfully must NOT emit"

skip_notice = build_sample_size_notice({
    "status": "skipped",
    "reason": "insufficient_runs",
    "min_required": MIN_RUNS_PER_FAULT,
})
assert skip_notice is not None, "skipped framework MUST emit a notice"
assert skip_notice["severity"] == "warning"

# Validate against the schema block
nb = _NoticeBlock.model_validate(skip_notice)
print(f"Threshold       : n >= {MIN_RUNS_PER_FAULT}")
print(f"Severity        : {nb.severity}")
print(f"Title           : {nb.title}")
print(f"Body length     : {len(nb.body)} chars")
print(f"First 120 chars : {nb.body[:120]}...")
print("\nAll Phase D builder assertions passed.")


Threshold       : n >= 30
Severity        : NoticeSeverity.warning
Title           : Inadequate sample size — statistical framework not applied
Body length     : 566 chars
First 120 chars : This certification executed n = 5 independent runs per fault category (15 runs total), which is below the framework-mand...

All Phase D builder assertions passed.


## Phase E — Hypothesis overlay (statistical findings)

When the upstream `statistical_hypothesis` block is `status: ok`, Phase E
overlays the certification report with two pieces of content:

* **Inline strips** appended after each metric chart-pair+table in §5–§6
  (TTD, TTM, tool-calls, detection rate, mitigation rate). Strips carry a
  `verdict`, deterministic per-category `facts[]` (numbers, CIs, STABLE /
  WIDE-CI tags), the canonical hypothesis `method` line, plus an
  LLM-generated `summary` (≤ 25 words) and `findings` paragraph.
* **Two dedicated sections** for H-03..H-09 (latency significance, RAI /
  security compliance, SLA, tail risk, temporal stability), inserted
  after §8 (Safety & Compliance).

When `statistical_hypothesis.status` is `skipped` or `not_requested`, the
overlay is fully suppressed and the report shape is unchanged — the §1
sample-size notice (Phase D) already explains why.

The cells below exercise:

1. The deterministic skeleton (no LLM) on a fully-populated `results` block.
2. The suppression path on a `status=skipped` block.
3. Schema validation of one synthesized `HypothesisStripBlock`.


In [17]:
import json
from types import SimpleNamespace

from cert_builder.scripts.narratives.hypothesis_overlay_builder import (
    build_overlay_skeleton,
)

# Load the upstream Phase 2 hypothesis-framework output. The wrapper exposes
# meta + validation + results; the overlay only needs the `results` payload.
_HYP_PATH = _REPO_ROOT / "hypothesis_framework" / "data" / "output" / "hypothesis_results.json"
_full = json.loads(_HYP_PATH.read_text(encoding="utf-8"))
ctx_ok = SimpleNamespace(
    statistical_hypothesis={"status": "ok", "results": _full["results"]}
)

overlay = build_overlay_skeleton(ctx_ok)

print(f"suppressed       : {overlay.suppressed}")
print(f"inline metrics   : {sorted(overlay.inline_strips.keys())}")
for metric_key, strips in overlay.inline_strips.items():
    s0 = strips[0]
    fact0 = s0["facts"][0]
    print(
        f"  {metric_key:<32s} verdict={s0['verdict']:<13s} "
        f"first-fact='{fact0['label']}: {fact0['text']}'"
    )

print("\nDedicated H-section block counts:")
for attr in (
    "h03_section_blocks", "h04_section_blocks", "h05_section_blocks",
    "h06_section_blocks", "h07_section_blocks", "h08_section_blocks",
    "h09_section_blocks",
):
    print(f"  {attr}: {len(getattr(overlay, attr))}")

# Sanity assertions
assert not overlay.suppressed
assert "time_to_detect" in overlay.inline_strips
assert "time_to_mitigate" in overlay.inline_strips
ttd0 = overlay.inline_strips["time_to_detect"][0]
assert ttd0["method"], "method line must be populated deterministically"
assert len(ttd0["facts"]) >= 3, "expect a fact per fault category"
print("\nDeterministic skeleton assertions passed.")


suppressed       : False
inline metrics   : ['fault_detection_success_rate', 'fault_mitigation_success_rate', 'time_to_detect', 'time_to_mitigate', 'tool_calls']
  time_to_detect                   verdict=pass          first-fact='Application: IQM 43 s, CI [35 s, 49 s] (±17% — STABLE)'
  time_to_mitigate                 verdict=pass          first-fact='Application: IQM 195 s, CI [175 s, 217 s] (±11% — STABLE)'
  tool_calls                       verdict=pass          first-fact='Application: IQM 15.0, CI [14.1, 15.9] (±6% — STABLE)'
  fault_detection_success_rate     verdict=flag          first-fact='Application: 86.7% (52/60), Wilson 95% CI [75.8%, 93.1%]'
  fault_mitigation_success_rate    verdict=flag          first-fact='Application: 66.7% (40/60), Wilson 95% CI [54.1%, 77.3%]'

Dedicated H-section block counts:
  h03_section_blocks: 3
  h04_section_blocks: 2
  h05_section_blocks: 3
  h06_section_blocks: 3
  h07_section_blocks: 3
  h08_section_blocks: 3
  h09_section_blocks: 3

Det

In [18]:
import importlib
import sys

# Drop any stale `schema.*` module the kernel may have cached from earlier
# cells (before the Phase E schema extension).
for _mod in list(sys.modules):
    if _mod == "schema" or _mod.startswith("schema."):
        del sys.modules[_mod]

import cert_builder.schema.certification_schema as _csm
importlib.reload(_csm)
_HypothesisStripBlock = _csm.HypothesisStripBlock

# 2a — suppressed path: status=skipped (e.g. insufficient_runs)
ctx_skip = SimpleNamespace(statistical_hypothesis={
    "status": "skipped",
    "reason": "insufficient_runs",
    "min_required": 30,
})
overlay_skip = build_overlay_skeleton(ctx_skip)
assert overlay_skip.suppressed and overlay_skip.suppression_reason == "insufficient_runs"
assert not overlay_skip.inline_strips
assert not overlay_skip.h03_section_blocks
print(f"skipped overlay -> suppressed={overlay_skip.suppressed} "
      f"reason={overlay_skip.suppression_reason!r}")

# 2b — not_requested path
ctx_nr = SimpleNamespace(statistical_hypothesis={"status": "not_requested"})
overlay_nr = build_overlay_skeleton(ctx_nr)
assert overlay_nr.suppressed and overlay_nr.suppression_reason == "not_requested"
print(f"not_requested  -> suppressed={overlay_nr.suppressed} "
      f"reason={overlay_nr.suppression_reason!r}")

# 2c — schema validation of one strip from the OK overlay
strip_dict = overlay.inline_strips["time_to_detect"][0]
hyp_strip = _HypothesisStripBlock.model_validate(strip_dict)
print(f"\nValidated H-01 strip for {hyp_strip.metric_label}:")
print(f"  verdict        : {hyp_strip.verdict}")
print(f"  facts          : {len(hyp_strip.facts)}")
print(f"  method         : {hyp_strip.method}")
print(f"  summary (stub) : {hyp_strip.summary}")
print(f"  findings       : {hyp_strip.findings!r}")
print("\nAll Phase E builder assertions passed.")


skipped overlay -> suppressed=True reason='insufficient_runs'
not_requested  -> suppressed=True reason='not_requested'

Validated H-01 strip for Time-to-Detect:
  verdict        : pass
  facts          : 3
  method         : IQM (25% trimmed mean) + Bootstrap BCa 95% CI, B = 10,000.
  summary (stub) : H-01 Time-to-Detect — verdict: pass.
  findings       : None

All Phase E builder assertions passed.
